# Module 090: Milestone Project - Automation Bot / Mini API

Phase 9: Databases & Web Apps | Duration: 3 hours | Milestone

## Setup: FastAPI App

In [ ]:
import logging
import os
import shutil
from pathlib import Path
from typing import Dict, List

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

app: FastAPI = FastAPI(title='Automation Bot')

class FileInfo(BaseModel):
    name: str
    size: int
    type: str
    path: str

class OrganizeRequest(BaseModel):
    directory: str

class OrganizeResponse(BaseModel):
    summary: Dict[str, int]
    total: int

class SystemInfo(BaseModel):
    os: str
    cpu_count: int
    cwd: str
    disk_usage: Dict[str, str]

## Helper Functions

In [ ]:
EXT_MAP: Dict[str, str] = {
    '.jpg': 'Images', '.jpeg': 'Images', '.png': 'Images',
    '.pdf': 'Documents', '.txt': 'Documents', '.doc': 'Documents',
    '.mp3': 'Audio', '.wav': 'Audio',
    '.zip': 'Archives', '.tar': 'Archives', '.gz': 'Archives',
    '.mp4': 'Videos', '.avi': 'Videos',
}

def get_file_type(filename: str) -> str:
    return EXT_MAP.get(Path(filename).suffix.lower(), 'Others')

## File Management Endpoints

In [ ]:
@app.get('/files', response_model=List[FileInfo])
def list_files(path: str = '.') -> List[FileInfo]:
    if not os.path.isdir(path):
        raise HTTPException(status_code=404, detail='Directory not found')
    entries: List[FileInfo] = []
    for entry in sorted(os.listdir(path)):
        full: str = os.path.join(path, entry)
        entries.append(FileInfo(
            name=entry,
            size=os.path.getsize(full) if os.path.isfile(full) else 0,
            type='file' if os.path.isfile(full) else 'directory',
            path=full,
        ))
    return entries

@app.post('/organize', response_model=OrganizeResponse)
def organize_dir(req: OrganizeRequest) -> OrganizeResponse:
    if not os.path.isdir(req.directory):
        raise HTTPException(status_code=404, detail='Directory not found')
    summary: Dict[str, int] = {}
    total: int = 0
    for f in os.listdir(req.directory):
        fp: str = os.path.join(req.directory, f)
        if os.path.isdir(fp):
            continue
        cat: str = get_file_type(f)
        cat_dir: str = os.path.join(req.directory, cat)
        os.makedirs(cat_dir, exist_ok=True)
        shutil.move(fp, os.path.join(cat_dir, f))
        summary[cat] = summary.get(cat, 0) + 1
        total += 1
    return OrganizeResponse(summary=summary, total=total)

## System Info Endpoint

In [ ]:
@app.get('/system-info', response_model=SystemInfo)
def system_info() -> SystemInfo:
    total, used, free = shutil.disk_usage('.')
    return SystemInfo(
        os=os.name,
        cpu_count=os.cpu_count() or 0,
        cwd=os.getcwd(),
        disk_usage={
            'total': f'{total / (1024**3):.1f} GB',
            'used': f'{used / (1024**3):.1f} GB',
            'free': f'{free / (1024**3):.1f} GB',
        },
    )

# Quick test
from fastapi.testclient import TestClient
client = TestClient(app)
print('System info:', client.get('/system-info').json())

## Testing Endpoints

In [ ]:
# Create a temp directory to test organize
import tempfile
tmpdir = tempfile.mkdtemp()
for fname in ['photo.jpg', 'doc.pdf', 'song.mp3', 'notes.txt']:
    Path(os.path.join(tmpdir, fname)).write_text('test')

resp = client.post('/organize', json={'directory': tmpdir})
print('Organize result:', resp.json())

resp2 = client.get(f'/files', params={'path': tmpdir})
print('Files after organize:', [f['name'] for f in resp2.json()])

shutil.rmtree(tmpdir)

## Next Steps

The complete solution with task scheduling, logging, and CLI interface is in `project/solution/automation_bot.py`. Open `project/starter_code.py` and implement the TODO functions.